# Fit action planning model

September 2025

In [1]:
import jax
import jax.numpy as jnp
import optax

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import model

sns.set_theme()

## Model fitting

<!-- Base model (H0): $p(a|c) \propto \exp(\alpha \cdot (-w_r \cdot c_{\text{risk}}(a) - w_e \cdot c_{\text{effort}}(a)))$

Relationship model (H1): $p(a|c) \propto \exp(\alpha \cdot (-w_d \cdot c_{\text{discomfort}}(a|c) - w_r \cdot c_{\text{risk}}(a) - w_e \cdot c_{\text{effort}}(a)))$ -->

### Load action planning data

In [ ]:
def load_data():
    """Load and prepare the planning data"""
    print("Loading planning data...")
    data = pd.read_csv("../data/planning-1/main_trials_tidy.csv")
    data.insert(
        data.columns.get_loc("scenario_label") + 2,
        "scenario_idx",
        data["scenario_label"].apply(lambda x: model.scenario_labels.index(x))
    )
    
    data_scenario_idx = jnp.array(data["scenario_idx"].values)
    data_action = jnp.array(data["action"].values)
    data_closeness = jnp.array(data["closeness"].values)
    data_p_action = jnp.array(data["p_action"].values)
    
    print(f"Loaded {len(data)} data points")
    return data, data_scenario_idx, data_action, data_closeness, data_p_action

In [3]:
data, data_scenario_idx, data_action, data_closeness, data_p_action = load_data()

Loading planning data...
Loaded 5696 data points


In [4]:
model.predict_risk_effort_vmap(data_scenario_idx, data_action, data_closeness, 1.0, 1.0, 1.0, 1.0)

Array([0.8532942 , 0.13291354, 0.00968558, ..., 0.05192333, 0.01738444,
       0.0023996 ], dtype=float32)

In [5]:
@jax.jit
def compute_NLL(preds, responses):
    epsilon = 1e-8
    preds_safe = jnp.clip(preds, epsilon, 1.0)
    responses_safe = jnp.clip(responses, epsilon, 1.0)
    nll = -jnp.sum(responses_safe * jnp.log(preds_safe))
    return nll

In [6]:
def fit_params(
    model_type,
    initial_params,
    data_scenario_idx,
    data_action,
    data_closeness,
    lr=0.001,
    tol=1e-6,
    max_steps=5000,
):
    """Fit model parameters using gradient descent"""
    predict_fn = model.get_vmap_predictor(model_type)

    def loss_fn(params):
        preds = predict_fn(data_scenario_idx, data_action, data_closeness, *params)
        return compute_NLL(preds, data_p_action)

    params = initial_params
    grad_fn = jax.value_and_grad(loss_fn)
    opt = optax.adam(learning_rate=lr)
    opt_state = opt.init(params)

    prev_nll = None 
    for step in range(max_steps):
        nll, grad = grad_fn(params)
        updates, opt_state = opt.update(grad, opt_state)
        params = optax.apply_updates(params, updates)

        params = params.at[:].set(jnp.clip(params[:], 0, jnp.inf)) # ensure params are positive

        if step % 1000 == 0:
            print(f"Step {step}, NLL: {nll}, params: {params}")

        if prev_nll is not None and nll > prev_nll:
            print(f"NLL increased at step {step}, stopping")
            break

        prev_nll = nll

    best_nll = loss_fn(params)

    return params, best_nll

In [ ]:
params_and_nlls = {}
for model_type in model.model_types:
    print(f"Fitting params for {model_type}")
    best_params, best_nll = fit_params(
        model_type=model_type,
        initial_params=jnp.array([1.0, 1.0, 1.0, 1.0]),
        data_scenario_idx=data_scenario_idx,
        data_action=data_action,
        data_closeness=data_closeness,
    )
    params_and_nlls[model_type] = (best_params, best_nll)
    print("----")
    print(
        f"Best params for {model_type}: alpha: {best_params[0]}, w_d: {best_params[1]}, w_r: {best_params[2]}, w_e: {best_params[3]}"
    )
    print(f"Best NLL for {model_type}: {best_nll}")
    print("----")

Fitting params for risk
Step 0, NLL: 2586.11767578125, params: [0.999 1.    0.999 1.   ]
Step 1000, NLL: 1920.675537109375, params: [0.49372387 1.         0.49372387 1.        ]
NLL increased at step 1725, stopping
----
Best params for risk: alpha: 0.4430391490459442, w_d: 1.0, w_r: 0.4430391490459442, w_e: 1.0
Best NLL for risk: 1916.74267578125
----
Fitting params for effort
Step 0, NLL: 2248.72021484375, params: [0.999 1.    1.    0.999]
Step 1000, NLL: 1971.96337890625, params: [0.50104403 1.         1.         0.50104403]
Step 2000, NLL: 1967.251220703125, params: [0.39404774 1.         1.         0.39404774]
NLL increased at step 2260, stopping
----
Best params for effort: alpha: 0.38429027795791626, w_d: 1.0, w_r: 1.0, w_e: 0.38429027795791626
Best NLL for effort: 1967.1734619140625
----
Fitting params for discomfort
Step 0, NLL: 2276.08251953125, params: [0.999 0.999 1.    1.   ]
Step 1000, NLL: 1900.526611328125, params: [0.5402815 0.5402815 1.        1.       ]
NLL increased 

Add model predictions to dataframe

In [ ]:
data_with_preds = data.copy()
for model_type in model.model_types:
    best_params, best_nll = params_and_nlls[model_type]
    predict_fn = model.get_vmap_predictor(model_type)
    preds = predict_fn(data_scenario_idx, data_action, data_closeness, *best_params)
    data_with_preds[f"{model_type}_pred"] = preds

# Save the updated dataframe
data_with_preds.to_csv("../data/planning-1/main_trials_tidy_with_preds.csv", index=False)
